In [6]:
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks

asr_inference_pipeline = pipeline(
    task=Tasks.auto_speech_recognition,
    model='iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch', model_revision="v2.0.4",
    # vad_model='iic/speech_fsmn_vad_zh-cn-16k-common-pytorch', vad_model_revision="v2.0.4",
    # punc_model='iic/punc_ct-transformer_zh-cn-common-vocab272727-pytorch', punc_model_revision="v2.0.4",
    # spk_model="iic/speech_campplus_sv_zh-cn_16k-common",
    # spk_model_revision="v2.0.2",
    device='cuda:1',
)

vad_inference_pipeline = pipeline(
    task=Tasks.voice_activity_detection,
    model='iic/speech_fsmn_vad_zh-cn-16k-common-pytorch',
    model_revision="v2.0.4",
    device='cuda:1',
)

whisper_large_inference_pipeline = pipeline(
    task=Tasks.auto_speech_recognition,
    model='iic/Whisper-large-v3', model_revision="v2.0.5",
    device='cuda:1',)

whisper_large_turbo_inference_pipeline = pipeline(
    task=Tasks.auto_speech_recognition,
    model='iic/Whisper-large-v3-turbo', model_revision="master",
    device='cuda:1',)

2025-03-22 06:18:54,682 - modelscope - INFO - Use user-specified model revision: v2.0.4
2025-03-22 06:18:55,062 - modelscope - INFO - initiate model from /home/zhangjiayuan/.cache/modelscope/hub/iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch
2025-03-22 06:18:55,063 - modelscope - INFO - initiate model from location /home/zhangjiayuan/.cache/modelscope/hub/iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch.
2025-03-22 06:18:55,064 - modelscope - INFO - initialize model from /home/zhangjiayuan/.cache/modelscope/hub/iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch


funasr version: 1.2.2.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
New version is available: 1.2.6.
Please use the command "pip install -U funasr" to upgrade.


KeyboardInterrupt: 

## 读取部分音频文件

In [3]:
def read_wav_scp(filename):
    utts = []
    utt2wav = {}
    with open(filename, 'r' ) as f:
        for line in f:
            line = line.strip()
            if line:
                utt, wav_path = line.split()
                utt2wav[utt] = wav_path
                utts.append(utt)
    
    return utts, utt2wav


def asr_paraformer_large(segs, speech_data, fs=16000):
    asr_res = {}
    asr_res['asr_model'] = 'iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch'
    asr_res['asr_model_vision'] = 'v2.0.4'
    asr_res['full_text'] = ""
    asr_res['segs'] = []
    
    full_text = ""
    for j, seg in enumerate(segs):
        st = int(seg[0]*fs/1000.0)
        ed = int(seg[1]*fs/1000.0)
        chunk = speech_data[st:ed]  
        res = asr_inference_pipeline(input=chunk,)
        text = res[0]['text']
        full_text += text
        
        asr_res['segs'].append({"seg": seg, "text": text})
        
    asr_res['full_text'] = full_text
    
    return asr_res

def asr_whisper_large(segs, speech_data, fs=16000):
    asr_res = {}
    asr_res['asr_model'] = 'iic/Whisper-large-v3'
    asr_res['asr_model_vision'] = 'v2.0.5'
    asr_res['full_text'] = ""
    asr_res['segs'] = []
    
    full_text = ""
    for j, seg in enumerate(segs):
        st = int(seg[0]*fs/1000.0)
        ed = int(seg[1]*fs/1000.0)
        chunk = speech_data[st:ed]  
        res = whisper_large_inference_pipeline(input=chunk)
        text = res[0]['text']
        full_text += text
        
        asr_res['segs'].append({"seg": seg, "text": text})
        
    asr_res['full_text'] = full_text
    
    return asr_res    


def asr_whisper_large_turbo(segs, speech_data, fs=16000):
    asr_res = {}
    asr_res['asr_model'] = 'iic/Whisper-large-v3-turbo'
    asr_res['asr_model_vision'] = 'master'
    asr_res['full_text'] = ""
    asr_res['segs'] = []
    
    full_text = ""
    for j, seg in enumerate(segs):
        st = int(seg[0]*fs/1000.0)
        ed = int(seg[1]*fs/1000.0)
        chunk = speech_data[st:ed]  
        res = whisper_large_turbo_inference_pipeline(input=chunk)
        text = res[0]['text']
        full_text += text
        
        asr_res['segs'].append({"seg": seg, "text": text})
        
    asr_res['full_text'] = full_text
    
    return asr_res 
        
    

In [ ]:
import json
import soundfile as sf 


wav_scp_file = '/data/nas/dataset/asr/kefu/shidian/wav1.scp'


utts, utt2wav = read_wav_scp(wav_scp_file)


data_annos = []
for i, utt in enumerate(utts):
    wav_path = utt2wav[utt]
    speech_data, fs = sf.read(wav_path)
    vad_res = vad_inference_pipeline(input=wav_path)
    data = {}
    data['utt'] = utt
    
    data['fs'] = fs
    data['dur'] = len(speech_data)/fs
    data['wav_path'] = wav_path
    data['annos'] = []  # 包含多个切分方式下的结果
    
    anno = {}
    anno['vad_model'] = 'iic/speech_fsmn_vad_zh-cn-16k-common-pytorch'
    anno['vad_model_vision'] = 'v2.0.4'
    anno['vad_res'] = vad_res[0]['value']
    anno['vad_asr_res'] = []    # 这个vad模型切分下的片段， 包含多个模型识别的结果
    
    vad_segs = vad_res[0]['value']
    asr_paraformer_res = asr_paraformer_large(vad_segs, speech_data, fs=fs)
    asr_whisper_large_res = asr_whisper_large(vad_segs, speech_data, fs=fs)
    asr_whisper_large_turbo_res = asr_whisper_large_turbo(vad_segs, speech_data, fs=fs)
    
    anno['vad_asr_res'].append(asr_paraformer_res)
    anno['vad_asr_res'].append(asr_whisper_large_res)
    anno['vad_asr_res'].append(asr_whisper_large_turbo_res)
    
    data['annos'].append(anno)
    
    data_annos.append(data)
    
    anno_json_file = "{}.anno.json".format(wav_path.replace(".wav", ""))
    with open(anno_json_file, 'w') as f:
        json.dump(data, f, ensure_ascii=False)

    
with open('./anno3.jsonl',  'w') as f:
    for data in data_annos:
        f.write(json.dumps(data, ensure_ascii=False) + '\n')

In [13]:
print(wav_path.replace(".wav", ""))

import os

print(os.path.dirname(wav_path))
print(data)

with open('a.json', 'w') as f:
    json.dump(data, f, ensure_ascii=False)

/data/nas/dataset/asr/kefu/shidian/wavs/date1211/cc-1866749004768968704
/data/nas/dataset/asr/kefu/shidian/wavs/date1211
{'utt': 'cc-1866749004768968704', 'fs': 16000, 'dur': 48.96, 'wav_path': '/data/nas/dataset/asr/kefu/shidian/wavs/date1211/cc-1866749004768968704.wav', 'annos': [{'vad_model': 'iic/speech_fsmn_vad_zh-cn-16k-common-pytorch', 'vad_model_vision': 'v2.0.4', 'vad_res': [[3820, 4500], [5070, 7110], [8030, 8640], [19210, 20180], [20460, 21120], [34530, 36620], [38410, 44420], [45090, 48940]], 'vad_asr_res': [{'asr_model': 'iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch', 'asr_model_vision': 'v2.0.4', 'full_text': '那我们业务员门口的那个里面买方便面靠着听用电板年才能拿到嗯你好我是圆通快递的你好呃有车反馈有个原套包裹业务员已经给您送达那个新街宾馆您收到了吗好那工单我们就做完结了啊尾号八四六幺好再见', 'segs': [{'seg': [3820, 4500], 'text': '那我们业务员'}, {'seg': [5070, 7110], 'text': '门口的那个里面买方便面靠着'}, {'seg': [8030, 8640], 'text': '听用电板'}, {'seg': [19210, 20180], 'text': '年才能拿到'}, {'seg': [20460, 21120], 'text': '嗯'}, {'seg': [34530, 36620], 'text'

# senceVoice


In [11]:
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks
import soundfile as sf
import re

wav_path = '/data/nas/dataset/asr/kefu/huaian/wavs/0008b15509a96de829597376315fd1aa@10.190.101.161_0.wav'
speech_data, fs = sf.read(wav_path)

vad_inference_pipeline = pipeline(
    task=Tasks.voice_activity_detection,
    model='iic/speech_fsmn_vad_zh-cn-16k-common-pytorch',
    model_revision="v2.0.4",
    device='cuda:1',
)

asr_inference_pipeline = pipeline(
    task=Tasks.auto_speech_recognition,
    model='iic/SenseVoiceSmall',
    model_revision="master",
    device="cuda:0",)






vad_res = vad_inference_pipeline(input=speech_data, disable_pbar=True)
segs = vad_res[0]['value']
print(segs)

results = []
for j, seg in enumerate(segs):
    st = int(seg[0]*fs/1000.0)
    ed = int(seg[1]*fs/1000.0)
    chunk = speech_data[st:ed]  
    res = asr_inference_pipeline(input=chunk, disable_pbar=True)
    results.append(res)
    print(res)



2025-03-22 06:34:52,066 - modelscope - INFO - Use user-specified model revision: v2.0.4
2025-03-22 06:34:52,332 - modelscope - INFO - initiate model from /home/zhangjiayuan/.cache/modelscope/hub/iic/speech_fsmn_vad_zh-cn-16k-common-pytorch
2025-03-22 06:34:52,333 - modelscope - INFO - initiate model from location /home/zhangjiayuan/.cache/modelscope/hub/iic/speech_fsmn_vad_zh-cn-16k-common-pytorch.
2025-03-22 06:34:52,334 - modelscope - INFO - initialize model from /home/zhangjiayuan/.cache/modelscope/hub/iic/speech_fsmn_vad_zh-cn-16k-common-pytorch


funasr version: 1.2.2.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel


2025-03-22 06:34:52,628 - modelscope - WARNING - No preprocessor field found in cfg.
2025-03-22 06:34:52,629 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2025-03-22 06:34:52,630 - modelscope - WARNING - Cannot find available config to build preprocessor at mode inference, current config: {'model_dir': '/home/zhangjiayuan/.cache/modelscope/hub/iic/speech_fsmn_vad_zh-cn-16k-common-pytorch'}. trying to build by task and model information.
2025-03-22 06:34:52,630 - modelscope - WARNING - No preprocessor key ('funasr', 'voice-activity-detection') found in PREPROCESSOR_MAP, skip building preprocessor.


New version is available: 1.2.6.
Please use the command "pip install -U funasr" to upgrade.


2025-03-22 06:34:53,309 - modelscope - WARNING - Using branch: master as version is unstable, use with caution
2025-03-22 06:34:53,669 - modelscope - INFO - initiate model from /home/zhangjiayuan/.cache/modelscope/hub/iic/SenseVoiceSmall
2025-03-22 06:34:53,670 - modelscope - INFO - initiate model from location /home/zhangjiayuan/.cache/modelscope/hub/iic/SenseVoiceSmall.
2025-03-22 06:34:53,671 - modelscope - INFO - initialize model from /home/zhangjiayuan/.cache/modelscope/hub/iic/SenseVoiceSmall


funasr version: 1.2.2.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
New version is available: 1.2.6.
Please use the command "pip install -U funasr" to upgrade.


2025-03-22 06:34:58,560 - modelscope - WARNING - No preprocessor field found in cfg.
2025-03-22 06:34:58,560 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2025-03-22 06:34:58,561 - modelscope - WARNING - Cannot find available config to build preprocessor at mode inference, current config: {'model_dir': '/home/zhangjiayuan/.cache/modelscope/hub/iic/SenseVoiceSmall'}. trying to build by task and model information.
2025-03-22 06:34:58,561 - modelscope - WARNING - No preprocessor key ('funasr', 'auto-speech-recognition') found in PREPROCESSOR_MAP, skip building preprocessor.


[[3090, 3860], [5750, 7500], [9450, 12550], [12830, 13970], [17870, 20080], [21090, 26440], [28800, 32080], [35650, 38780], [41670, 43550], [44030, 45980]]
[{'key': 'rand_key_1t9EwL56nGisi', 'text': '<|zh|><|EMO_UNKNOWN|><|Speech|><|woitn|>嗯'}]
[{'key': 'rand_key_WgNZq6ITZM5jt', 'text': '<|zh|><|NEUTRAL|><|Speech|><|woitn|>呀好高我是怎么个班呢'}]
[{'key': 'rand_key_gUe52RvEJgwBu', 'text': '<|zh|><|NEUTRAL|><|Speech|><|woitn|>你好单号是尾号六幺三八的对吗'}]
[{'key': 'rand_key_NO6n9JEC3HqdZ', 'text': '<|zh|><|NEUTRAL|><|Speech|><|woitn|>寄往江西'}]
[{'key': 'rand_key_6J6afU1zT0YQO', 'text': '<|zh|><|NEUTRAL|><|Speech|><|woitn|>你是邓邓先生对吧'}]
[{'key': 'rand_key_aNF03vpUuT3em', 'text': '<|zh|><|NEUTRAL|><|Speech|><|woitn|>邓先生取价吗我现在给您补发了一份发到您的手机上了您看一下手机短信啊'}]
[{'key': 'rand_key_6KopZ9jZICffu', 'text': '<|zh|><|NEUTRAL|><|Speech|><|woitn|>对您现在看一下不要挂您看一下有收到短信吗'}]
[{'key': 'rand_key_4G7FgtJsThJv0', 'text': '<|zh|><|NEUTRAL|><|Speech|><|woitn|>刚刚手机响了一下您翻看一下应该已经收到了'}]
[{'key': 'rand_key_7In9ZMJLsCfMZ', 'text': '<|zh|><|NEUTRA

In [13]:
import re

import re

def parse_speech_string(s: str) -> dict:

    # 使用正则提取所有标签内容
    tags = re.findall(r'<\|(.*?)\|>', s)
    
    # 提取最后一个标签之后的文本内容
    text = s.split('|>')[-1].strip()
    
    language = tags[0] if len(tags) > 0 else ""
    emotion = tags[1] if len(tags) > 1 else ""
    event = tags[2] if len(tags) > 2 else ""
    itn = tags[3] if len(tags) > 3 else ""
    
    res = {}
    res['language'] = language
    res['emotion'] = emotion
    res['event'] = event
    res['itn'] = itn
    res['text'] = text
    
    return res


for res in results:
    sv_txt = res[0]['text']
    a = parse_speech_string(sv_txt)
    print(a)

{'language': 'zh', 'emotion': 'EMO_UNKNOWN', 'event': 'Speech', 'itn': 'woitn', 'text': '嗯'}
{'language': 'zh', 'emotion': 'NEUTRAL', 'event': 'Speech', 'itn': 'woitn', 'text': '呀好高我是怎么个班呢'}
{'language': 'zh', 'emotion': 'NEUTRAL', 'event': 'Speech', 'itn': 'woitn', 'text': '你好单号是尾号六幺三八的对吗'}
{'language': 'zh', 'emotion': 'NEUTRAL', 'event': 'Speech', 'itn': 'woitn', 'text': '寄往江西'}
{'language': 'zh', 'emotion': 'NEUTRAL', 'event': 'Speech', 'itn': 'woitn', 'text': '你是邓邓先生对吧'}
{'language': 'zh', 'emotion': 'NEUTRAL', 'event': 'Speech', 'itn': 'woitn', 'text': '邓先生取价吗我现在给您补发了一份发到您的手机上了您看一下手机短信啊'}
{'language': 'zh', 'emotion': 'NEUTRAL', 'event': 'Speech', 'itn': 'woitn', 'text': '对您现在看一下不要挂您看一下有收到短信吗'}
{'language': 'zh', 'emotion': 'NEUTRAL', 'event': 'Speech', 'itn': 'woitn', 'text': '刚刚手机响了一下您翻看一下应该已经收到了'}
{'language': 'zh', 'emotion': 'NEUTRAL', 'event': 'Speech', 'itn': 'woitn', 'text': '好还有其他问题吗'}
{'language': 'zh', 'emotion': 'HAPPY', 'event': 'Speech', 'itn': 'woitn', 'text': '好拜次